In [ ]:
import kagglehub


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# Download latest version
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)
#df = df.drop(columns="Unnamed: 0", axis=1)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()



In [ ]:
# Task 1: Write your code here:


def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
     print(f"Encoding column: {col}")
     df[col] =df[col].fillna(df[col].mode()[0])
number_cols = df.select_dtypes(include=["number"]).drop("Target",axis=1).columns

for col in number_cols:
     print(f"Encoding column: {col}")
     df[col] =df[col].fillna(df[col].mean())
check_missing_values(df)


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoders = {}
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Target")
#Target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score,f1_score


from catboost import CatBoostClassifier

# Single model:

models = {

  "CatBoost": CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)
}

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'f1': [] }

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred_test = model.predict(X_test)



    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")




    results[model_name]['accuracy'].append(accuracy)
    results[model_name]['f1'].append(f1)
for model_name in results:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")


In [ ]:
# Task 1: Write your code here:
catboost_model = models["CatBoost"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:
print(features[0])

In [ ]:
# # Task Bonus: Write your code here:
X_new = df["P_2"]
y_new = df['Target']
 # Single model:

models = {

   "CatBoost": CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)
 }

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results_new = {}

for model_name in models:
  results_new[model_name] = {'accuracy_new': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X_new,y_new )):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred_test_new = model.predict(X_test)



    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred_test_new)
    # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")




    results_new[model_name]['accuracy_new'].append(accuracy)

for model_name in results_new:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  accuracy_new:  {np.mean(results_new[model_name]['accuracy_new']):.4f}")


